# Seed Qdrant without Docker

This notebook seeds `hue_cultural_data.json` into embedded local Qdrant at `data/qdrant`. It does not need Docker or a Qdrant server.

In [3]:
from pathlib import Path
import os
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src" / "bevietnam" / "scripts" / "hue_cultural_data.json").exists():
            return candidate
        if candidate.name == "scripts" and (candidate / "hue_cultural_data.json").exists():
            return candidate.parents[2]
    raise FileNotFoundError("Could not find BeVietnam repo root.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
PACKAGE_ROOT = REPO_ROOT / "src" / "bevietnam"
DATA_PATH = PACKAGE_ROOT / "scripts" / "hue_cultural_data.json"
REQUIREMENTS = PACKAGE_ROOT / "ai" / "requirements.txt"

for path in (REPO_ROOT, PACKAGE_ROOT):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

print("Repo root:", REPO_ROOT)
print("Data file:", DATA_PATH)
print("Requirements:", REQUIREMENTS)

Repo root: /home/phuckhang/MyWorkspace/TDTT/BeVietnam
Data file: /home/phuckhang/MyWorkspace/TDTT/BeVietnam/src/bevietnam/scripts/hue_cultural_data.json
Requirements: /home/phuckhang/MyWorkspace/TDTT/BeVietnam/src/bevietnam/ai/requirements.txt


Run this dependency cell once for the selected notebook kernel. It can take a while because `sentence-transformers` pulls the embedding stack.

Configure embedded Qdrant. The app can read the same local database later if it runs with the same `QDRANT_PATH` value.

In [4]:
import importlib

os.environ["QDRANT_PATH"] = str(REPO_ROOT / "data" / "qdrant")

config_module = importlib.import_module("src.bevietnam.ai.common.config")
config_module = importlib.reload(config_module)

store_module = importlib.import_module("ai.common.qdrant_store")
store_module = importlib.reload(store_module)

settings = config_module.settings
qdrant_store = store_module.qdrant_store

print("QDRANT_PATH:", settings.qdrant_path)
print("Collection:", settings.qdrant_collection)

ModuleNotFoundError: No module named 'qdrant_client'

Load the seed data, create the collection if needed, embed, and upsert.

In [ ]:
import json

with open(DATA_PATH, encoding="utf-8") as f:
    documents = json.load(f)

print(f"Loaded {len(documents)} documents")
qdrant_store.ensure_collection()
count = qdrant_store.upsert_documents(documents)
print(f"Seeded {count} documents")

Quick search check.

In [ ]:
results = qdrant_store.search("history of Hue", limit=3)
results